In [ ]:
# ------------------------
# Dummy baseline for Kaggle submission
# Generates random multi-label predictions
# ------------------------
import os
import csv
import random
from tqdm import tqdm

# --- Paths ---
TEST_DIR = "Amazon_products/test"  # modify if needed
TEST_CORPUS_PATH = os.path.join(TEST_DIR, "test_corpus.txt")  # product_id \t text
SUBMISSION_PATH = "submission.csv"  # output file

# --- Constants ---
NUM_CLASSES = 531  # total number of classes (0–530)
MIN_LABELS = 1     # minimum number of labels per sample
MAX_LABELS = 3     # maximum number of labels per sample

# --- Load test corpus ---
def load_corpus(path):
    """Load test corpus into {pid: text} dictionary."""
    pid2text = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t", 1)
            if len(parts) == 2:
                pid, text = parts
                pid2text[pid] = text
    return pid2text

pid2text_test = load_corpus(TEST_CORPUS_PATH)
pid_list_test = list(pid2text_test.keys())

# --- Generate random predictions ---
all_pids, all_labels = [], []
for pid in tqdm(pid_list_test, desc="Generating dummy predictions"):
    n_labels = random.randint(MIN_LABELS, MAX_LABELS)
    labels = random.sample(range(NUM_CLASSES), n_labels)
    labels = sorted(labels)
    all_pids.append(pid)
    all_labels.append(labels)

# --- Save submission file ---
with open(SUBMISSION_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["pid", "labels"])
    for pid, labels in zip(all_pids, all_labels):
        writer.writerow([pid, ",".join(map(str, labels))])

print(f"Dummy submission file saved to: {SUBMISSION_PATH}")
print(f"Total samples: {len(all_pids)}, Classes per sample: {MIN_LABELS}-{MAX_LABELS}")

Generating dummy predictions: 100%|██████████| 19658/19658 [00:00<00:00, 388263.46it/s]

Dummy submission file saved to: submission.csv
Total samples: 19658, Classes per sample: 1-3


여기부터 MAGIC START!

In [ ]:
# Step 0. Imports & Paths
import os
from pathlib import Path
from collections import defaultdict, deque

import numpy as np
from tqdm import tqdm

import torch
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModel

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)


In [ ]:
# 프로젝트 루트 기준 경로 설정 (dummy_baseline와 같은 레벨에 Amazon_products가 있다고 가정)
ROOT = Path(".")          # 지금 노트북이 dummy_baseline/ 안에 있을 때
DATA_DIR = ROOT / "Amazon_products"

TRAIN_PATH = DATA_DIR / "train" / "train_corpus.txt"
TEST_PATH  = DATA_DIR / "test" / "test_corpus.txt"
CLASSES_PATH = DATA_DIR / "classes.txt"
HIER_PATH = DATA_DIR / "class_hierarchy.txt"
KEYWORDS_PATH = DATA_DIR / "class_related_keywords.txt"

OUT_DIR = Path(".")  # silver label, embedding 저장 위치 (dummy_baseline/)
OUT_DIR.mkdir(exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
# Step 1.1 Load class names, hierarchy, and keywords

def load_classes(path: Path):
    id2name = {}
    name2id = {}
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            cid_str, cname = line.split("\t", 1)
            cid = int(cid_str)
            cname = cname.strip()
            id2name[cid] = cname
            name2id[cname] = cid
    return id2name, name2id


def load_hierarchy(path: Path, num_classes: int):
    parents = defaultdict(list)
    children = defaultdict(list)
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            p_str, c_str = line.split("\t")
            p = int(p_str); c = int(c_str)
            parents[c].append(p)
            children[p].append(c)

    # depth 계산 (root는 parent가 없는 노드)
    depth = {cid: None for cid in range(num_classes)}
    roots = [cid for cid in range(num_classes) if len(parents[cid]) == 0]

    q = deque()
    for r in roots:
        depth[r] = 0
        q.append(r)

    while q:
        u = q.popleft()
        for v in children[u]:
            if depth[v] is None:
                depth[v] = depth[u] + 1
                q.append(v)

    return parents, children, depth, roots

def load_keywords(path: Path, name2id):
    id2keywords = defaultdict(list)
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # 콜론 없는 라인은 (예: 'swings') 그냥 스킵
            if ":" not in line:
                continue
            cname, kws_str = line.split(":", 1)
            cname = cname.strip()
            if cname not in name2id:
                # classes.txt에 없는 이름이면 건너뜀
                continue
            cid = name2id[cname]
            kw_list = [w.strip() for w in kws_str.split(",") if w.strip()]
            id2keywords[cid] = kw_list
    return id2keywords

id2name, name2id = load_classes(CLASSES_PATH)
num_classes = len(id2name)
print("num_classes:", num_classes)

id2keywords = load_keywords(KEYWORDS_PATH, name2id)
parents, children, depth, roots = load_hierarchy(HIER_PATH, num_classes)


print("roots:", roots[:10], "...")
print("example class 0:", id2name[0], id2keywords.get(0, []), "depth:", depth[0])


num_classes: 531
roots: [0, 3, 10, 23, 40, 169] ...
example class 0: grocery_gourmet_food ['snacks', 'condiments', 'beverages', 'specialty_foods', 'spices', 'cooking_oils', 'baking_ingredients', 'gourmet_chocolates', 'artisanal_cheeses', 'organic_foods'] depth: 0


In [ ]:
# Step 1.2 Load train/test corpus

def load_corpus(path: Path):
    """train_corpus.txt / test_corpus.txt에서 (pid, text)를 줄 단위로 읽기"""
    pids, texts = [], []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            pid, txt = line.split("\t", 1)
            pids.append(pid)
            texts.append(txt)
    return pids, texts

train_pids, train_texts = load_corpus(TRAIN_PATH)
test_pids,  test_texts  = load_corpus(TEST_PATH)

print("train docs:", len(train_pids))
print("test docs:", len(test_pids))
print("sample doc:", train_pids[0], train_texts[0][:200], "...")


train docs: 29487
test docs: 19658
sample doc: 0 omron hem 790it automatic blood pressure monitor with advanced omron health management software so far this machine has worked well and is very simple to use . it is nice to have immediate feedback on ...


In [ ]:
# Step 1.3 Build textual descriptions for each class (name + keywords)

def build_class_texts(id2name, id2keywords):
    texts = []
    for cid in range(len(id2name)):
        name = id2name[cid]
        kws = id2keywords.get(cid, [])
        text = name
        if kws:
            text += ": " + ", ".join(kws)
        texts.append(text)
    return texts

class_texts = build_class_texts(id2name, id2keywords)
print(class_texts[0])


grocery_gourmet_food: snacks, condiments, beverages, specialty_foods, spices, cooking_oils, baking_ingredients, gourmet_chocolates, artisanal_cheeses, organic_foods


In [ ]:
# Step 1.4 Encode documents and classes with BERT

MODEL_NAME = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert = AutoModel.from_pretrained(MODEL_NAME).to(device)
bert.eval()  # 학습은 나중 단계에서

@torch.no_grad()
def encode_texts(texts, batch_size=32, max_length=128):
    """리스트[str] → [N, hidden_size] 텐서 (CLS 임베딩)"""
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i : i + batch_size]
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(device)
        outputs = bert(**enc)
        cls = outputs.last_hidden_state[:, 0, :]  # [B, H]
        all_embs.append(cls.cpu())
    return torch.cat(all_embs, dim=0)  # [N, H]


In [ ]:
# 실제 임베딩 계산 (시간 조금 걸릴 수 있음)
train_embs = encode_texts(train_texts, batch_size=32, max_length=128)
class_embs = encode_texts(class_texts, batch_size=64, max_length=32)

print("train_embs:", train_embs.shape)
print("class_embs:", class_embs.shape)

# 필요하다면 디스크에 저장해두면 2단계에서 재사용 가능
torch.save({"pids": train_pids, "embeddings": train_embs}, OUT_DIR / "train_bert_emb.pt")
torch.save({"class_embs": class_embs}, OUT_DIR / "class_bert_emb.pt")


100%|██████████| 9/9 [00:00<00:00, 22.03it/s]


train_embs: torch.Size([29487, 768])
class_embs: torch.Size([531, 768])


In [ ]:
# Step 1.5 Compute document–class cosine similarity

def cosine_sim_matrix(doc_embs: torch.Tensor, class_embs: torch.Tensor):
    # 정규화 후 matmul
    d = F.normalize(doc_embs, dim=1)
    c = F.normalize(class_embs, dim=1)
    return d @ c.T  # [N_docs, N_classes]

sims = cosine_sim_matrix(train_embs, class_embs)  # float32
print("sims shape:", sims.shape)
print("sample sims for doc 0 (top 5):")
top5 = torch.topk(sims[0], k=5)
for score, cid in zip(top5.values, top5.indices):
    print(float(score), int(cid), id2name[int(cid)])


sims shape: torch.Size([29487, 531])
sample sims for doc 0 (top 5):
0.8460230827331543 440 stimulants
0.8392992615699768 303 occupational_physical_therapy_aids
0.8375481367111206 298 outdoor_safety
0.8334247469902039 369 tea_gifts
0.8323081731796265 418 mints


In [ ]:
# 1.6 계층 구조를 이용한 silver label 생성 (조상 depth 제한 + 배치 계산, CPU)

import torch
import torch.nn.functional as F
from collections import defaultdict

num_docs, emb_dim = train_embs.shape
num_classes = class_embs.size(0)

print("num_docs:", num_docs, "emb_dim:", emb_dim, "num_classes:", num_classes)

# ---- ancestors 미리 계산 ----
def compute_ancestors(parents_dict, cid):
    anc = set()
    stack = list(parents_dict[cid])
    while stack:
        u = stack.pop()
        if u in anc:
            continue
        anc.add(u)
        stack.extend(parents_dict[u])
    return anc

if "ancestors" not in globals():
    ancestors = {cid: compute_ancestors(parents, cid) for cid in range(num_classes)}

# ---- silver label 생성 파라미터 ----
TOP_K = 5              # core 후보 top-k
BASE_TAU = 0.7         # best similarity 대비 상대 임계값
MIN_DEPTH_CORE = 0          # core 에서 depth 필터 없애기 (root도 후보)
MIN_ANCESTOR_DEPTH = 1      # root(depth=0)만 빼고, 그 위는 다 포함

# 결과 텐서: -1(ignore), 0(negative), 1(positive)
silver_labels = torch.full(
    (num_docs, num_classes),
    -1,
    dtype=torch.long
)

# 메모리 절약을 위해 CPU에서 batch 단위로 similarity 계산
train_embs_cpu = train_embs.cpu()
class_embs_cpu = class_embs.cpu()
all_ids = set(range(num_classes))

BATCH_SIZE = 512  # 메모리 여유에 따라 256, 1024 등 조절 가능

for start in range(0, num_docs, BATCH_SIZE):
    end = min(start + BATCH_SIZE, num_docs)
    batch = train_embs_cpu[start:end]          # [B, H]

    # [B, 1, H] vs [1, C, H] → [B, C]
    with torch.no_grad():
        sims_batch = F.cosine_similarity(
            batch.unsqueeze(1),
            class_embs_cpu.unsqueeze(0),
            dim=-1
        )   # [B, C]

    B = sims_batch.size(0)
    for bi in range(B):
        i = start + bi
        s = sims_batch[bi]  # [C]

        # 1) 유사도 top-k 후보
        topk = torch.topk(s, k=min(TOP_K, num_classes))
        cand_ids = topk.indices.tolist()
        cand_vals = topk.values.tolist()
        best_sim = float(cand_vals[0])

        # 2) core class 선택
        core = []
        for cid, val in zip(cand_ids, cand_vals):
            cid = int(cid)
            # 너무 상위 노드는 core로 쓰지 않음
            if depth.get(cid, 0) <= MIN_DEPTH_CORE:
                continue
            # best 대비 너무 낮은 유사도는 제외
            if val < BASE_TAU * best_sim:
                continue
            core.append(cid)

        if not core:
            # 아무도 안 남으면 top-1 이라도 하나 선택
            core = [int(cand_ids[0])]

        # 3) positive set = core + (깊이가 어느 정도 있는 조상)
        pos_set = set()
        for cid in core:
            pos_set.add(cid)
            for anc in ancestors[cid]:
                if depth.get(anc, 0) >= MIN_ANCESTOR_DEPTH:
                    pos_set.add(anc)

        neg_set = all_ids - pos_set

        silver_labels[i, list(pos_set)] = 1
        silver_labels[i, list(neg_set)] = 0

    print(f"processed docs {start} ~ {end-1} / {num_docs}")

silver_labels.shape

num_docs: 29487 emb_dim: 768 num_classes: 531
processed docs 0 ~ 511 / 29487
processed docs 512 ~ 1023 / 29487
processed docs 1024 ~ 1535 / 29487
processed docs 1536 ~ 2047 / 29487
processed docs 2048 ~ 2559 / 29487
processed docs 2560 ~ 3071 / 29487
processed docs 3072 ~ 3583 / 29487
processed docs 3584 ~ 4095 / 29487
processed docs 4096 ~ 4607 / 29487
processed docs 4608 ~ 5119 / 29487
processed docs 5120 ~ 5631 / 29487
processed docs 5632 ~ 6143 / 29487
processed docs 6144 ~ 6655 / 29487
processed docs 6656 ~ 7167 / 29487
processed docs 7168 ~ 7679 / 29487
processed docs 7680 ~ 8191 / 29487
processed docs 8192 ~ 8703 / 29487
processed docs 8704 ~ 9215 / 29487
processed docs 9216 ~ 9727 / 29487
processed docs 9728 ~ 10239 / 29487
processed docs 10240 ~ 10751 / 29487
processed docs 10752 ~ 11263 / 29487
processed docs 11264 ~ 11775 / 29487
processed docs 11776 ~ 12287 / 29487
processed docs 12288 ~ 12799 / 29487
processed docs 12800 ~ 13311 / 29487
processed docs 13312 ~ 13823 / 29487

torch.Size([29487, 531])

In [ ]:
# Step 1.7 Save silver labels and some stats

torch.save(
    {
        "pids": train_pids,
        "silver_labels": silver_labels,  # int8 tensor, values in {1, 0, -1}
    },
    OUT_DIR / "train_silver_labels.pt",
)

# ---- 통계 출력 ----
num_pos = int((silver_labels == 1).sum().item())
num_neg = int((silver_labels == 0).sum().item())
num_ign = int((silver_labels == -1).sum().item())
print("Positive labels:", num_pos)
print("Negative labels:", num_neg)
print("Ignored labels:", num_ign)
print("Avg positives per doc:", num_pos / num_docs)

# 샘플 몇 개 찍어보기
for idx in range(3):
    text = train_texts[idx][:200].replace("\n", " ")
    y = silver_labels[idx]
    pos_cids = torch.nonzero(y == 1, as_tuple=False).view(-1).tolist()
    print("\nDoc:", train_pids[idx])
    print("Text:", text, "...")
    print("Silver classes:",
          [(cid, id2name[cid]) for cid in pos_cids])


Positive labels: 267683
Negative labels: 15389914
Ignored labels: 0
Avg positives per doc: 9.0780004747855

Doc: 0
Text: omron hem 790it automatic blood pressure monitor with advanced omron health management software so far this machine has worked well and is very simple to use . it is nice to have immediate feedback on ...
Silver classes: [(38, 'health_care'), (74, 'medical_supplies_equipment'), (135, 'safety'), (208, 'gourmet_gifts'), (265, 'candy_chocolate'), (298, 'outdoor_safety'), (303, 'occupational_physical_therapy_aids'), (369, 'tea_gifts'), (418, 'mints'), (440, 'stimulants')]

Doc: 1
Text: natural factors whey factors chocolate works well , but there is a lot of dead space in the container when you first open it up . the container comes 3 4 4 5 full and the rest in empty space . ...
Silver classes: [(208, 'gourmet_gifts'), (255, 'breads_bakery'), (265, 'candy_chocolate'), (271, 'snack_food'), (355, 'trail_mix'), (366, 'popcorn'), (373, 'cheese_gifts'), (418, 'mints'), (480, 

2단계 시작!1

In [ ]:
# GCN용 adjacency (양방향 + self-loop + D^{-1/2} A D^{-1/2})
A = torch.zeros(num_classes, num_classes, dtype=torch.float32)
for p in range(num_classes):
    for c in children[p]:
        A[p, c] = 1.0
        A[c, p] = 1.0

A = A + torch.eye(num_classes)  # self-loop

deg = A.sum(dim=1)
deg_inv_sqrt = torch.where(deg > 0, deg.pow(-0.5), torch.zeros_like(deg))
D_inv_sqrt = torch.diag(deg_inv_sqrt)
A_hat = D_inv_sqrt @ A @ D_inv_sqrt   # [C, C]

# device로 옮기기
A_hat = A_hat.to(device)
class_embs = class_embs.to(device)
train_embs = train_embs.to(device)
silver_labels = silver_labels.to(device)

A_hat.shape


torch.Size([531, 531])

In [ ]:
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Step 2.2 Dataset & DataLoader

class SilverDataset(Dataset):
    def __init__(self, doc_embs, labels):
        self.doc_embs = doc_embs
        self.labels = labels  # [N, C], values in {1,0,-1}

    def __len__(self):
        return self.doc_embs.size(0)

    def __getitem__(self, idx):
        return self.doc_embs[idx], self.labels[idx]

N, emb_dim = train_embs.shape
val_ratio = 0.1
n_val = int(N * val_ratio)
n_train = N - n_val

perm = torch.randperm(N)
train_idx = perm[:n_train]
val_idx = perm[n_train:]

train_ds = SilverDataset(train_embs[train_idx], silver_labels[train_idx])
val_ds   = SilverDataset(train_embs[val_idx],   silver_labels[val_idx])

batch_size = 256
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

len(train_loader), len(val_loader)


(104, 12)

In [ ]:
# Step 2.3 LabelGCN model

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, X, A_hat):
        # X: [C, in_dim]
        H = A_hat @ X          # [C, in_dim]
        H = self.linear(H)     # [C, out_dim]
        return H

class LabelGCN(nn.Module):
    """
    - 클래스 초기 임베딩: class_embs (BERT + 키워드)
    - 2-layer GCN으로 클래스 임베딩 업데이트 (hierarchy 반영)
    - 문서 임베딩을 projection 한 후, 클래스 임베딩과 dot-product로 로짓 계산
    """
    def __init__(self, num_classes, emb_dim, hidden_dim, A_hat, class_init):
        super().__init__()
        self.num_classes = num_classes
        self.emb_dim = emb_dim

        # 그래프/초기 임베딩은 buffer로 등록 (학습 X)
        self.register_buffer("A_hat", A_hat)
        self.register_buffer("class_init", class_init)

        self.gcn1 = GCNLayer(emb_dim, hidden_dim)
        self.gcn2 = GCNLayer(hidden_dim, emb_dim)

        self.doc_proj = nn.Linear(emb_dim, emb_dim)
        self.dropout = nn.Dropout(0.2)

    def get_class_embeddings(self):
        X = self.class_init
        H = F.relu(self.gcn1(X, self.A_hat))
        H = self.dropout(H)
        H = self.gcn2(H, self.A_hat)
        return H    # [C, emb_dim]

    def forward(self, doc_embs):
        # doc_embs: [B, emb_dim]
        class_embs = self.get_class_embeddings()          # [C, emb_dim]
        doc_h = self.dropout(F.relu(self.doc_proj(doc_embs)))  # [B, emb_dim]
        logits = doc_h @ class_embs.T                     # [B, C]
        return logits

model = LabelGCN(
    num_classes=num_classes,
    emb_dim=emb_dim,
    hidden_dim=256,
    A_hat=A_hat,
    class_init=class_embs,
).to(device)

sum(p.numel() for p in model.parameters()) / 1e6


0.984832

In [ ]:
# Step 2.4 Loss function & evaluation

bce = nn.BCEWithLogitsLoss(reduction="none")

def multilabel_loss(logits, labels):
    """
    logits: [B, C]
    labels: [B, C] in {1,0,-1}; -1은 ignore
    """
    mask = (labels != -1).float()    # [B, C]
    if mask.sum() == 0:
        return torch.tensor(0.0, device=logits.device, requires_grad=True)

    target = (labels == 1).float()
    loss_raw = bce(logits, target)   # [B, C]
    loss = (loss_raw * mask).sum() / mask.sum()
    return loss

@torch.no_grad()
def eval_loss(model, loader):
    model.eval()
    total, cnt = 0.0, 0.0
    for X, Y in loader:
        X, Y = X.to(device), Y.to(device)
        logits = model(X)
        mask = (Y != -1).float()
        if mask.sum() == 0:
            continue
        target = (Y == 1).float()
        loss_raw = bce(logits, target)
        total += (loss_raw * mask).sum().item()
        cnt += mask.sum().item()
    return total / max(cnt, 1.0)


In [ ]:

# Step 2.5 First-stage training with silver labels

def train_model(model, train_loader, val_loader, epochs=10, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    best_val = float("inf")
    best_state = None

    for ep in range(1, epochs + 1):
        model.train()
        total, cnt = 0.0, 0.0
        for X, Y in train_loader:
            X, Y = X.to(device), Y.to(device)
            logits = model(X)
            loss = multilabel_loss(logits, Y)

            opt.zero_grad()
            loss.backward()
            opt.step()

            with torch.no_grad():
                mask = (Y != -1).float()
                if mask.sum() > 0:
                    target = (Y == 1).float()
                    total += (bce(logits, target) * mask).sum().item()
                    cnt += mask.sum().item()

        train_l = total / max(cnt, 1.0)
        val_l = eval_loss(model, val_loader)
        print(f"[Epoch {ep:02d}] train_loss={train_l:.4f}  val_loss={val_l:.4f}")

        if val_l < best_val:
            best_val = val_l
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    return model

model = train_model(model, train_loader, val_loader, epochs=10, lr=1e-3)


[Epoch 01] train_loss=0.1706  val_loss=0.0567
[Epoch 02] train_loss=0.0539  val_loss=0.0468
[Epoch 03] train_loss=0.0495  val_loss=0.0442
[Epoch 04] train_loss=0.0474  val_loss=0.0426
[Epoch 05] train_loss=0.0452  val_loss=0.0404
[Epoch 06] train_loss=0.0436  val_loss=0.0390
[Epoch 07] train_loss=0.0419  val_loss=0.0376
[Epoch 08] train_loss=0.0407  val_loss=0.0387
[Epoch 09] train_loss=0.0400  val_loss=0.0353
[Epoch 10] train_loss=0.0392  val_loss=0.0352


In [ ]:

# Step 2.6 Self-training: refine labels once and retrain

@torch.no_grad()
def refine_labels(model, doc_embs, labels,
                  pos_thr=0.9, neg_thr=0.1):
    """
    doc_embs: [N, H]
    labels  : [N, C] in {1,0,-1}
    return  : new_labels in {1,0,-1}
    """
    model.eval()
    N, C = labels.shape
    new_labels = torch.full_like(labels, -1)  # 기본은 ignore

    batch_size = 256
    prob_list = []

    for i in range(0, N, batch_size):
        X = doc_embs[i:i+batch_size].to(device)
        logits = model(X)
        probs = torch.sigmoid(logits).cpu()
        prob_list.append(probs)

    probs_all = torch.cat(prob_list, dim=0)   # [N, C]

    # 이미 positive였거나, 확률이 충분히 높은 건 1
    pos_mask = (labels == 1) | (probs_all > pos_thr)
    # 확실히 negative로 보이는 건 0
    neg_mask = (labels == 0) & (probs_all < neg_thr)

    new_labels[pos_mask] = 1
    new_labels[neg_mask] = 0
    # 나머지는 -1 (ignore)

    return new_labels

refined_labels = refine_labels(model, train_embs.cpu(), silver_labels.cpu(),
                               pos_thr=0.9, neg_thr=0.1).to(device)

pos_cnt = (refined_labels == 1).sum().item()
neg_cnt = (refined_labels == 0).sum().item()
ign_cnt = (refined_labels == -1).sum().item()
print("Refined labels -> pos:", pos_cnt, "neg:", neg_cnt, "ignore:", ign_cnt)


Refined labels -> pos: 268120 neg: 15086290 ignore: 303187


In [ ]:
train_ds2 = SilverDataset(train_embs[train_idx], refined_labels[train_idx])
val_ds2   = SilverDataset(train_embs[val_idx],   refined_labels[val_idx])

train_loader2 = DataLoader(train_ds2, batch_size=batch_size, shuffle=True)
val_loader2   = DataLoader(val_ds2, batch_size=batch_size, shuffle=False)

model = train_model(model, train_loader2, val_loader2, epochs=5, lr=5e-4)


[Epoch 01] train_loss=0.0309  val_loss=0.0207
[Epoch 02] train_loss=0.0236  val_loss=0.0188
[Epoch 03] train_loss=0.0222  val_loss=0.0178
[Epoch 04] train_loss=0.0212  val_loss=0.0172
[Epoch 05] train_loss=0.0203  val_loss=0.0172


In [1]:
# optionial
# Step 2.7 Save final model (for step 3 prediction)

# %%
torch.save(
    {
        "state_dict": model.state_dict(),
        "A_hat": A_hat.cpu(),
        "class_init": class_embs.cpu(),
        "emb_dim": emb_dim,
        "num_classes": num_classes,
    },
    OUT_DIR / "labelgcn_final.pt",
)
print("Saved model to labelgcn_final.pt")


NameError: name 'torch' is not defined

여기부터 3단계!!!

In [ ]:
# Step 3.1 Encode test corpus with BERT

test_embs = encode_texts(test_texts, batch_size=32, max_length=128)
test_embs = test_embs.to(device)

print("test_embs:", test_embs.shape)


100%|██████████| 615/615 [01:00<00:00, 10.18it/s]

test_embs: torch.Size([19658, 768])


In [ ]:
# Step 3.2 Predict per-class probabilities for test docs

@torch.no_grad()
def predict_probs(model, doc_embs, batch_size=256):
    model.eval()
    all_probs = []
    for i in range(0, doc_embs.size(0), batch_size):
        X = doc_embs[i : i + batch_size].to(device)
        logits = model(X)
        probs = torch.sigmoid(logits).cpu()
        all_probs.append(probs)
    return torch.cat(all_probs, dim=0)

test_probs = predict_probs(model, test_embs)
print("test_probs:", test_probs.shape)


test_probs: torch.Size([19658, 531])


In [ ]:
# Step 3.3 Convert probabilities to final labels
#  - leaf 노드 우선
#  - min_prob 이상만 후보
#  - parent/child 같이 나오면 더 구체적인 쪽만
#  - 최종 2~3개 라벨 출력

N_test, C = test_probs.shape

# ancestors 없으면 한 번 계산
from collections import defaultdict, deque

def compute_ancestors(parents_dict, cid):
    anc = set()
    stack = list(parents_dict[cid])
    while stack:
        u = stack.pop()
        if u in anc:
            continue
        anc.add(u)
        stack.extend(parents_dict[u])
    return anc

if "ancestors" not in globals():
    ancestors = {cid: compute_ancestors(parents, cid) for cid in range(num_classes)}

# leaf 노드(자식이 없는 클래스) 집합
leaf_ids = {cid for cid in range(num_classes) if len(children[cid]) == 0}

TOP_K_CAND = 15   # 확률 상위 몇 개까지 후보로 볼지
MIN_DEPTH   = 1   # 너무 상위(depth<=1) 카테고리는 가급적 제외
MIN_LABELS  = 2
MAX_LABELS  = 3
MIN_PROB    = 0.25  # 이 확률보다 낮으면 1차 후보에서 제외

pred_labels_per_doc = []

for i in range(N_test):
    probs = test_probs[i]  # [C]
    topk = torch.topk(probs, k=min(TOP_K_CAND, C))
    cand_ids = topk.indices.tolist()

    selected = []

    # 1차: leaf + min_prob 조건을 만족하는 것 위주로 선택
    for cid in cand_ids:
        cid = int(cid)
        p = float(probs[cid])

        # 최소 확률 조건
        if p < MIN_PROB:
            continue

        # 너무 상위 노드는 스킵
        if depth.get(cid, 0) <= MIN_DEPTH:
            continue

        # leaf 우선: leaf가 아니면 일단 건너뜀
        if cid not in leaf_ids:
            continue

        # 부모-자식 관계 처리: 더 구체적인 쪽만 남기기
        conflict = False
        for s in list(selected):
            if s in ancestors[cid]:
                # 기존 s가 cid의 조상이면 s를 제거하고 cid만 남김
                selected.remove(s)
            elif cid in ancestors[s]:
                # cid가 s의 조상이면 cid는 너무 상위 → 버림
                conflict = True
                break
        if conflict:
            continue

        selected.append(cid)
        if len(selected) == MAX_LABELS:
            break

    # leaf 위주로 골랐는데 2개가 안 되면, leaf 제약을 완화해서 후보로 채우기
    if len(selected) < MIN_LABELS:
        for cid in cand_ids:
            cid = int(cid)
            if cid in selected:
                continue
            # 너무 상위는 여전히 제외
            if depth.get(cid, 0) <= MIN_DEPTH:
                continue
            selected.append(cid)
            if len(selected) == MIN_LABELS:
                break

    # 최종 정리
    selected = sorted(set(selected))
    pred_labels_per_doc.append(selected)

# sanity check
for i in range(5):
    print("id:", test_pids[i],
          "labels:", pred_labels_per_doc[i],
          " (len =", len(pred_labels_per_doc[i]), ")")


id: 0 labels: [340, 396, 418]  (len = 3 )
id: 1 labels: [146, 340, 393]  (len = 3 )
id: 2 labels: [146, 164, 284]  (len = 3 )
id: 3 labels: [396, 418, 511]  (len = 3 )
id: 4 labels: [340, 369, 418]  (len = 3 )


In [ ]:
# Step 3.4 Build submission DataFrame and save as CSV

import csv

SUBMISSION_PATH = "2021320312_final.csv"

with open(SUBMISSION_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "label"])

    for pid, labels in zip(test_pids, pred_labels_per_doc):
        # pred_labels_per_doc[i] 는 정수 리스트 형태
        label_str = ",".join(str(x) for x in labels)
        writer.writerow([pid, label_str])

print(f"Saved submission file to: {SUBMISSION_PATH}")
print(f"Total samples: {len(pred_labels_per_doc)}  (each should have {MIN_LABELS}-{MAX_LABELS} labels)")

Saved submission file to: 2021320312_final.csv
Total samples: 19658  (each should have 2-3 labels)
